# Student Dropout Prediction - Preprocessing

This notebook prepares the final MVP dataset. It removes `Enrolled`, encodes the target, keeps the fixed 10 enrollment/background features, and saves `processed.csv`.

In [1]:
# Import libraries used for preprocessing.
from pathlib import Path
import json

import pandas as pd

## Load Data

Resolve project paths and load the raw UCI student dropout dataset.

In [2]:
# Resolve paths so the notebook works from either the project root or the notebooks folder.
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Feature config path:", FEATURE_CONFIG_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)

Project root: e:\Projects\student-dropout-prediction-ml
Dataset path: e:\Projects\student-dropout-prediction-ml\data\raw\dataset.csv
Feature config path: e:\Projects\student-dropout-prediction-ml\app\feature_config.json
Processed data path: e:\Projects\student-dropout-prediction-ml\data\processed\processed.csv


In [3]:
# Load raw data and feature config.
df = pd.read_csv(DATA_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

print("Raw dataset shape:", df.shape)
print("Raw target distribution:")
print(df["Target"].value_counts())

df.head()

Raw dataset shape: (4424, 35)
Raw target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Final MVP Feature Scope

The same 10 features are used from preprocessing through model training and the Streamlit app. Semester variables, macroeconomic fields, application mode/order, and post-acceptance/admin variables are excluded to reduce leakage and keep the MVP form understandable.

In [4]:
# Define final candidate MVP features from app/feature_config.json.
candidate_mvp_features = feature_config["features"]

print("Final MVP feature count:", len(candidate_mvp_features))
print(candidate_mvp_features)

Final MVP feature count: 10
['Marital status', 'Course', 'Previous qualification', "Mother's qualification", "Father's qualification", 'Displaced', 'Educational special needs', 'Gender', 'Age at enrollment', 'International']


## Filter Rows and Encode Target

`Enrolled` is removed because the project is a binary classification task: `Graduate` vs `Dropout`. Dropout is encoded as the positive class.

In [5]:
# Remove Enrolled and encode the target.
df_processed = df[df["Target"] != "Enrolled"].copy()

target_mapping = {
    "Graduate": 0,
    "Dropout": 1
}

df_processed["Target"] = df_processed["Target"].map(target_mapping)

print("Dataset shape after removing Enrolled:", df_processed.shape)
print("Encoded target distribution:")
print(df_processed["Target"].value_counts().sort_index())

Dataset shape after removing Enrolled: (3630, 35)
Encoded target distribution:
Target
0    2209
1    1421
Name: count, dtype: int64


## Keep Final Columns

The saved processed dataset contains only the 10 fixed MVP features plus the encoded target.

In [6]:
# Keep only selected features and target.
missing_features = [
    feature for feature in candidate_mvp_features
    if feature not in df_processed.columns
]

if missing_features:
    raise ValueError(f"Missing MVP features from dataset: {missing_features}")

df_processed = df_processed[candidate_mvp_features + ["Target"]].copy()

print("Processed dataset shape:", df_processed.shape)
df_processed.head()

Processed dataset shape: (3630, 11)


,Marital status,Course,Previous qualification,Mother's qualification,Father's qualification,Displaced,Educational special needs,Gender,Age at enrollment,International,Target
0,1,2,1,13,10,1,0,1,20,0,1
1,1,11,1,1,3,1,0,1,19,0,0
2,1,5,1,22,27,1,0,1,19,0,1
3,1,15,1,23,27,1,0,0,20,0,0
4,2,3,1,22,28,0,0,0,45,0,0


## Save Processed Dataset

This file is the source for model training and app input options.

In [7]:
# Save final processed dataset.
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

print("Processed dataset saved to:", PROCESSED_DATA_PATH)
print("Saved shape:", df_processed.shape)

Processed dataset saved to: e:\Projects\student-dropout-prediction-ml\data\processed\processed.csv
Saved shape: (3630, 11)


## Preprocessing Summary

- `Enrolled` rows are removed.
- Target is encoded as `Graduate = 0`, `Dropout = 1`.
- Final output has 10 MVP features plus `Target`.
- No semester, macroeconomic, application mode/order, or post-acceptance/admin variables are kept.